# Berry–Esseen: n25 B4: fresh INITIAL and all 100 resource panels

Select **Runtime -> Run all**. No command or parameter edits are required.
Authorize Google Drive when requested. Keep one session per selected task.

This notebook recomputes the full B4 INITIAL proof and resource acquisition,
then runs both unchanged native acceptors, including their corruption controls.
It does not wait for the four missing B4 count trees. These computations can run in parallel.
The imported STRONGER24 bound, analytic same-law implications and final count join
remain separate premises. A fresh regional result is not a global theorem.

Each of the four stages has its own two-hour CPU ceiling and external wall limit.
Live evidence is backed up every minute. Interrupted internal computations are not
automatically treated as accepted prefixes.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, time
CODE_COMMIT = "30ed60e64362ed5876daaf8ab01bbdf54c9d7d1c"
TASK = "n25-b4-parents"
REPOSITORY = "https://github.com/rodziewiczk/berry-esseen-end2end.git"
CODE = Path("/content") / ("berry-esseen-code-" + CODE_COMMIT[:12])
print("Fetching the pinned source commit:", CODE_COMMIT, flush=True)
if not CODE.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPOSITORY, str(CODE)], check=True)
    subprocess.run(["git", "-C", str(CODE), "checkout", "--quiet", "--detach", CODE_COMMIT], check=True)
observed = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
if observed != CODE_COMMIT:
    raise RuntimeError("Source commit mismatch")
subprocess.run(["git", "-C", str(CODE), "diff", "--exit-code", "HEAD", "--"], check=True)
# A rerun in an existing Colab kernel must not reuse modules from an older pin.
for module_name in tuple(sys.modules):
    if module_name == "be_replay" or module_name.startswith("be_replay."):
        del sys.modules[module_name]
sys.path.insert(0, str(CODE))
from be_replay.environment import snapshot
if not Path(sys.modules["be_replay"].__file__).resolve().is_relative_to(CODE.resolve()):
    raise RuntimeError("Imported execution code does not belong to the pinned checkout")
print(json.dumps(snapshot(), indent=2), flush=True)
print("Source code and environment recorded. Mathematical acceptance has not yet been established.", flush=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DURABLE = Path("/content/drive/MyDrive/Berry-Esseen-end2end")
DURABLE.mkdir(parents=True, exist_ok=True)
print("Logs and checkpoints:", DURABLE, flush=True)


In [ ]:
from datetime import datetime, timezone
SELFTEST = DURABLE / ("selftest-" + TASK + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".log")
print("Testing the supervisor on the actual Linux runtime. These lightweight fixtures do not verify the mathematics.", flush=True)
with SELFTEST.open("w", encoding="utf-8") as log:
    child = subprocess.Popen([sys.executable, "-B", "-S", "-m", "unittest", "discover",
                              "-s", "tests", "-p", "test_linux_supervisor.py", "-v"],
                             cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in child.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    code = child.wait()
if code != 0:
    raise RuntimeError("Supervisor self-tests failed; computation will not start. Log: " + str(SELFTEST))
print("Supervisor self-tests passed. Starting the selected calculation.", flush=True)


In [ ]:
print("Checking this job's orchestration and source-binding fixtures. No mathematics is executed by these tests.", flush=True)
DRIVER_SELFTEST = SELFTEST.with_name(SELFTEST.stem + "-driver.log")
with DRIVER_SELFTEST.open("w", encoding="utf-8") as log:
    result = subprocess.run([sys.executable, "-B", "-S", "-m", "unittest", "discover",
                             "-s", "tests", "-p", "test_n25_parents.py", "-v"],
                            cwd=CODE, stdout=log, stderr=subprocess.STDOUT, text=True)
fixture_lines = DRIVER_SELFTEST.read_text(encoding="utf-8").splitlines()
if result.returncode == 0:
    summary = next((line for line in reversed(fixture_lines) if line.startswith("Ran ")), "All selected fixtures completed.")
    print("Driver infrastructure fixtures: PASS.", summary, flush=True)
else:
    print("\n".join(fixture_lines), flush=True)
print("Infrastructure fixture log:", DRIVER_SELFTEST, flush=True)
if result.returncode != 0:
    raise RuntimeError("The selected driver failed its infrastructure fixtures; computation will not start.")


In [ ]:
from be_replay.n25_parents import run_colab
RESULT = run_colab('b4', durable_root=DURABLE)


In [ ]:
from IPython.display import display, FileLink
from google.colab import files
print("Execution status:", RESULT["status"]["state"], flush=True)
print("The global proof remains unverified.", flush=True)
print("Results ZIP on Google Drive:", RESULT["bundle"], flush=True)
display(FileLink(RESULT["bundle"]))
try:
    files.download(RESULT["bundle"])
except Exception as error:
    print("Download the ZIP from the Google Drive path shown above:", error, flush=True)
